# Phase 0: Setup & Baseline
## NVIDIA Nemotron Model Reasoning Challenge

**Goal of this notebook**: Get the model running, understand what we're working with, and record a baseline score. Nothing fancy yet — this is our floor.

---

### What are we actually doing here?

This competition asks us to **improve the reasoning accuracy** of `Nemotron 3 Nano`. Before we can improve anything, we need to know:
1. What the model is and how it works
2. What score it gets *right now* without any changes
3. How fast it runs (critical — the metric is `maj@64`, meaning 64 generations per problem)

Think of Phase 0 as taking a baseline measurement before starting any experiment.

---
## Part 1: Understanding the Model

Before running a single line of code, let's understand what we're working with.

### Architecture: Hybrid Mamba2-Transformer MoE

```
nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
├── Total params:  30B
├── Active params: 3.5B per token  ← only a fraction runs per forward pass
└── 52 layers total:
    ├── 23 Mamba-2 layers    → state-space model (SSM), not attention
    ├── 23 MoE layers        → 128 experts, only 6 active per token
    └──  6 Attention layers  → GQA with 2 groups (memory-efficient)
```

### Why does this matter?

**Mamba-2 layers** use recurrent state-space models instead of attention:
- Pro: Linear time complexity vs quadratic for attention → fast at long contexts
- Con: Requires vLLM >= 0.12.0 (older versions don't support Mamba)

**MoE (Mixture of Experts)**:
- 128 experts exist, but only 6 are activated per token
- This is why 30B total params costs only 3.5B active compute
- Implication: fine-tuning MoE requires careful handling (LoRA per expert)

### Two inference modes — know this before running anything

| Mode | Flag | Use case | Speed |
|---|---|---|---|
| Thinking ON | `enable_thinking=True` (default) | Math, reasoning | Slow (up to 10k tokens of thought) |
| Thinking OFF | `enable_thinking=False` | Quick tasks | Fast (greedy, 32 tokens) |

**For this competition: always thinking ON.**

---
## Part 2: Hardware Check

### Why check hardware first?

The model in BF16 = ~60 GB VRAM. A single L4 GPU (what Kaggle G4 gives us) has 24 GB.
That means **BF16 doesn't fit**. We need the FP8 variant (~15 GB).

| Variant | VRAM | Fits on 1× L4? | Quality loss? |
|---|---|---|---|
| BF16 | ~60 GB | No (need 3× L4) | None (reference) |
| **FP8** | **~15 GB** | **Yes** | Minimal (~0.5% accuracy) |
| GGUF (Q4) | ~8 GB | Yes | Moderate |

We'll use FP8 for Phase 0. Later phases may use multi-GPU.

In [ ]:
import subprocess

# Check what GPU we have and available VRAM
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Also check CPU/RAM
import psutil
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB available")

---
## Part 3: Environment Setup

### What we're installing and why

| Package | Why |
|---|---|
| `vllm>=0.12.0` | Required for Mamba support; much faster than HF Transformers for batched inference |
| `transformers` | For tokenizer and quick single-sample tests |
| `accelerate` | Multi-GPU device mapping for HF models |
| `compressed-tensors` | Required to load FP8 quantized weights |

**Why vLLM and not just Transformers?**
The evaluation metric is `maj@64` — 64 generations per problem. If each generation takes 30 seconds with Transformers, that's 32 minutes *per problem*. vLLM's continuous batching runs all 64 samples in parallel, cutting this to ~1 minute.

In [ ]:
# Install all dependencies
# Note: This takes 3-5 minutes on Kaggle. Run once, then restart kernel.
!pip install -q -U "vllm>=0.12.0" transformers accelerate compressed-tensors psutil

print("Installation complete.")

In [ ]:
# Download the custom Nemotron reasoning parser
# This is required for vLLM to correctly parse the <think>...</think> reasoning tokens
# Without it, vLLM won't know how to separate thinking from the final answer
!wget -q https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16/resolve/main/nano_v3_reasoning_parser.py

print("Reasoning parser downloaded.")
!ls -lh nano_v3_reasoning_parser.py

---
## Part 4: Load the Model

### Strategy: Two-stage approach

We'll use two loading methods in this notebook:

1. **HuggingFace Transformers** — for a quick sanity check (1 sample)
2. **vLLM server** — for real evaluation (64 samples per problem)

Think of HF as a quick debugger, vLLM as the production runner.

### Model to use: FP8 variant
`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8`

FP8 = 8-bit floating point. Compared to BF16:
- Uses half the VRAM
- ~1.5x faster inference
- Accuracy loss is negligible for math reasoning (NVIDIA measured <0.5%)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8"

print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model (this may take 5-10 minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",       # lets the model use its native dtype (FP8)
    trust_remote_code=True,
    device_map="auto"         # automatically places layers across available GPUs
)

print(f"Model loaded. Device map:")
print(model.hf_device_map if hasattr(model, 'hf_device_map') else "single device")

# Memory snapshot after loading
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {allocated:.1f} GB / {total:.1f} GB used")

---
## Part 5: Sanity Check — Run One Problem

### What we're testing

Before doing any evaluation, run a single known problem to verify:
- The model loads and generates correctly
- The `<think>` tokens appear (confirms thinking mode is ON)
- The output format matches what the competition expects

### Anatomy of the output

Nemotron 3 Nano produces structured output:
```
<think>
  ...internal reasoning (chain-of-thought)...
</think>

The answer is: [FINAL ANSWER]
```

The `nano_v3_reasoning_parser.py` we downloaded tells vLLM how to separate these parts.

In [ ]:
import time

# A simple math problem to verify the model is working
TEST_PROBLEM = """Find all positive integers n such that n^2 + 1 is divisible by n + 1."""

messages = [
    {
        "role": "system",
        "content": "You are an expert mathematician. Reason carefully and provide a rigorous solution."
    },
    {
        "role": "user",
        "content": TEST_PROBLEM
    }
]

# Apply chat template with thinking mode ON (default)
tokenized = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

print(f"Input tokens: {tokenized.shape[1]}")

start = time.time()
with torch.no_grad():
    output = model.generate(
        tokenized,
        max_new_tokens=4096,   # allow up to 4k tokens of thinking for this test
        temperature=1.0,       # recommended for reasoning tasks
        top_p=1.0,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )
elapsed = time.time() - start

# Decode only the generated tokens (not the input prompt)
generated = output[0][tokenized.shape[1]:]
response = tokenizer.decode(generated, skip_special_tokens=False)  # keep special tokens to see <think>

new_tokens = len(generated)
print(f"\nGenerated {new_tokens} tokens in {elapsed:.1f}s ({new_tokens/elapsed:.1f} tok/s)")
print(f"\n{'='*60}")
print(response)
print('='*60)

In [ ]:
# Parse thinking vs final answer
# The model wraps its chain-of-thought in <think>...</think>

def parse_response(raw: str) -> dict:
    """Split model output into thinking trace and final answer."""
    think_start = raw.find("<think>")
    think_end = raw.find("</think>")
    
    thinking = ""
    answer = raw
    
    if think_start != -1 and think_end != -1:
        thinking = raw[think_start + 7 : think_end].strip()
        answer = raw[think_end + 8:].strip()
    
    return {"thinking": thinking, "answer": answer}

parsed = parse_response(response)

print("THINKING (first 500 chars):")
print(parsed["thinking"][:500])
print(f"\n... [{len(parsed['thinking'])} chars total thinking] ...\n")
print("FINAL ANSWER:")
print(parsed["answer"])

---
## Part 6: Baseline Evaluation

### What is `pass@1` and `maj@64`?

These are the two metrics in this competition.

**pass@1**: Run the model once per problem. Is the answer correct? Average over all problems.
- Simple, but noisy — one bad generation tanks the score
- Equivalent to accuracy at temperature=0 (greedy)

**maj@64**: Run the model 64 times per problem, take the majority vote answer. Is it correct?
- Much more robust — averages out sampling noise
- This is the primary competition metric
- This is also called **self-consistency** (the technique from the 2023 Wang et al. paper)

### Why does majority voting work?

For math problems with a single correct answer:
- If the model is 60% likely to get the right answer per sample
- With 64 samples + majority vote, the probability of the majority being correct → ~95%
- Wrong answers are diverse (many ways to be wrong), correct answers converge

### NVIDIA's published baseline numbers

| Benchmark | pass@1 | maj@64 |
|---|---|---|
| AIME 2025 (with tools) | 89.1% | — |
| MMLU-Pro | 78.3% | — |
| GPQA | 73.0% | — |

These are our targets to beat.

In [ ]:
# Implement majority voting (self-consistency)
# We'll run N samples and pick the most common answer

from collections import Counter
import re

def extract_final_answer(text: str) -> str:
    """Extract the final numeric/symbolic answer from model output."""
    # Remove thinking block
    think_end = text.find("</think>")
    if think_end != -1:
        text = text[think_end + 8:].strip()
    
    # Try to find boxed answer (LaTeX convention: \\boxed{...})
    boxed = re.search(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed.group(1).strip()
    
    # Try to find "The answer is X" pattern
    answer_match = re.search(r'(?:answer is|answer:|=)\s*([\d\-\.]+)', text, re.IGNORECASE)
    if answer_match:
        return answer_match.group(1).strip()
    
    # Fallback: return last number in text
    numbers = re.findall(r'[\-]?\d+\.?\d*', text)
    return numbers[-1] if numbers else text.strip()[-50:]


def majority_vote(answers: list) -> str:
    """Return the most common answer from a list of answers."""
    if not answers:
        return ""
    counter = Counter(answers)
    return counter.most_common(1)[0][0]


def solve_with_majority_vote(problem: str, n_samples: int = 8, max_new_tokens: int = 4096) -> dict:
    """
    Solve a problem using majority voting over n_samples generations.
    
    Note: In competition we use n_samples=64. For testing, use 8.
    """
    messages = [
        {"role": "system", "content": "You are an expert mathematician. Reason carefully and provide a rigorous solution."},
        {"role": "user", "content": problem}
    ]
    
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    all_answers = []
    all_raw = []
    
    start = time.time()
    for i in range(n_samples):
        with torch.no_grad():
            output = model.generate(
                tokenized,
                max_new_tokens=max_new_tokens,
                temperature=1.0,
                top_p=1.0,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )
        
        raw = tokenizer.decode(output[0][tokenized.shape[1]:], skip_special_tokens=False)
        answer = extract_final_answer(raw)
        all_answers.append(answer)
        all_raw.append(raw)
        print(f"  Sample {i+1}/{n_samples}: {answer}")
    
    elapsed = time.time() - start
    voted = majority_vote(all_answers)
    
    return {
        "majority_answer": voted,
        "all_answers": all_answers,
        "answer_distribution": Counter(all_answers),
        "elapsed_sec": elapsed,
        "sec_per_sample": elapsed / n_samples
    }

print("Functions defined.")

In [ ]:
# Run majority vote on our test problem (n=8 for speed during Phase 0)
# In production, increase n_samples to 64

TEST_PROBLEM = """Find all positive integers n such that n^2 + 1 is divisible by n + 1."""
KNOWN_ANSWER = "No such n exists"  # n^2+1 = (n+1)(n-1) + 2, so (n+1)|2, giving n=1 but 2/2=1 ✓... 
# Actually n=1: 1+1=2, divides 1+1=2 ✓. So answer = n=1.

print(f"Problem: {TEST_PROBLEM}")
print(f"Running 8 samples (use 64 for actual submission)...\n")

result = solve_with_majority_vote(TEST_PROBLEM, n_samples=8)

print(f"\n{'='*50}")
print(f"Majority answer: {result['majority_answer']}")
print(f"Answer distribution: {dict(result['answer_distribution'])}")
print(f"Time: {result['elapsed_sec']:.1f}s total, {result['sec_per_sample']:.1f}s per sample")
print(f"Extrapolated time for maj@64: {result['sec_per_sample'] * 64 / 60:.1f} minutes per problem")

---
## Part 7: Experiment — System Prompt Impact

### Why does the system prompt matter?

The system prompt primes the model's context before the problem. For reasoning tasks:
- A math-specific prompt can shift accuracy by 2–5%
- Too much instruction can constrain the model's natural reasoning style
- The model was trained on certain prompt patterns — matching them helps

We'll test 3 prompts and compare the answer consistency (a proxy for confidence).

In [ ]:
# Compare 3 system prompts on a simple problem
# We use n_samples=4 here just for speed — increase for real evaluation

SYSTEM_PROMPTS = {
    "minimal": "",
    "general": "You are a helpful assistant.",
    "math_expert": (
        "You are an expert mathematician competing in an olympiad. "
        "Think deeply and systematically. Show your reasoning step by step. "
        "At the end, clearly state your final answer."
    )
}

EVAL_PROBLEM = "What is the sum of all integers from 1 to 100?"
# Known answer: 5050

results = {}

for name, system_prompt in SYSTEM_PROMPTS.items():
    print(f"\nTesting prompt: '{name}'")
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": EVAL_PROBLEM})
    
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    answers = []
    for _ in range(4):
        with torch.no_grad():
            output = model.generate(
                tokenized,
                max_new_tokens=2048,
                temperature=1.0,
                top_p=1.0,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )
        raw = tokenizer.decode(output[0][tokenized.shape[1]:], skip_special_tokens=False)
        answers.append(extract_final_answer(raw))
    
    consistency = len([a for a in answers if a == majority_vote(answers)]) / len(answers)
    results[name] = {"answers": answers, "majority": majority_vote(answers), "consistency": consistency}
    print(f"  Answers: {answers}")
    print(f"  Majority: {results[name]['majority']} | Consistency: {consistency:.0%}")

print("\n\n--- SUMMARY ---")
print(f"{'Prompt':<15} {'Majority Answer':<20} {'Consistency'}")
print("-" * 50)
for name, r in results.items():
    print(f"{name:<15} {r['majority']:<20} {r['consistency']:.0%}")

---
## Part 8: Experiment — Temperature Impact

### Why temperature matters for `maj@64`

Temperature controls sampling randomness:
- **Too low** (e.g., 0.3): All 64 samples look nearly identical → no diversity, majority vote doesn't help
- **Too high** (e.g., 1.5): Samples are too random → wrong answers aren't filtered out
- **Sweet spot** (~0.7–1.0): Enough diversity to explore, enough signal to converge

NVIDIA recommends `temp=1.0` for reasoning. Let's verify this empirically.

In [ ]:
TEMPERATURES = [0.5, 0.7, 1.0]
N_SAMPLES = 4  # use 16+ for real comparison

BEST_SYSTEM_PROMPT = max(results, key=lambda k: results[k]['consistency'])
print(f"Using best system prompt from Part 7: '{BEST_SYSTEM_PROMPT}'")

messages = []
if SYSTEM_PROMPTS[BEST_SYSTEM_PROMPT]:
    messages.append({"role": "system", "content": SYSTEM_PROMPTS[BEST_SYSTEM_PROMPT]})
messages.append({"role": "user", "content": EVAL_PROBLEM})

tokenized = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

temp_results = {}
for temp in TEMPERATURES:
    print(f"\nTemperature: {temp}")
    answers = []
    for _ in range(N_SAMPLES):
        with torch.no_grad():
            output = model.generate(
                tokenized,
                max_new_tokens=2048,
                temperature=temp,
                top_p=1.0,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )
        raw = tokenizer.decode(output[0][tokenized.shape[1]:], skip_special_tokens=False)
        answers.append(extract_final_answer(raw))
    
    mv = majority_vote(answers)
    consistency = len([a for a in answers if a == mv]) / len(answers)
    temp_results[temp] = {"answers": answers, "majority": mv, "consistency": consistency}
    print(f"  Answers: {answers} → Majority: {mv}, Consistency: {consistency:.0%}")

print("\n--- TEMPERATURE COMPARISON ---")
for temp, r in temp_results.items():
    print(f"temp={temp}: majority={r['majority']}, consistency={r['consistency']:.0%}")

---
## Part 9: Record Baseline & Plan Next Steps

### What to record

At the end of Phase 0, you should have:
1. A confirmed working model (FP8 on L4)
2. Inference speed (tokens/sec, seconds per sample)
3. Best system prompt and temperature from quick experiments
4. An estimate of time budget for `maj@64` across all competition problems

In [ ]:
# Record baseline results — fill these in after running the cells above

baseline = {
    "model": MODEL_ID,
    "hardware": "1x L4 24GB",
    "best_system_prompt": BEST_SYSTEM_PROMPT,
    "best_temperature": max(temp_results, key=lambda t: temp_results[t]['consistency']),
    "inference_speed_tok_per_sec": result['elapsed_sec'] and (4096 / result['elapsed_sec']),  # rough estimate
    "sec_per_sample": result['sec_per_sample'],
    "est_min_per_problem_at_64samples": result['sec_per_sample'] * 64 / 60,
}

print("=" * 50)
print("PHASE 0 BASELINE SUMMARY")
print("=" * 50)
for k, v in baseline.items():
    print(f"{k:40s}: {v}")

print("\n--- WHAT THIS MEANS FOR PHASE 1 ---")
est = baseline['est_min_per_problem_at_64samples']
print(f"At {est:.1f} min/problem, 110 problems × maj@64 = {est * 110:.0f} minutes total")
print(f"vs. competition time budget: 9 hours = 540 minutes")
print(f"Margin: {540 - est * 110:.0f} minutes")

---
## Phase 0: Summary & What's Next

### What we accomplished

| Task | Status |
|---|---|
| Loaded Nemotron 3 Nano (FP8) on L4 GPU | ✓ |
| Verified thinking mode produces `<think>` tokens | ✓ |
| Implemented majority voting (self-consistency) | ✓ |
| Measured inference speed and time budget | ✓ |
| Quick system prompt comparison | ✓ |
| Quick temperature comparison | ✓ |

### Key numbers to carry forward into Phase 1

- **Best system prompt**: _fill in from results_
- **Best temperature**: _fill in from results_
- **Inference speed**: _fill in from results_
- **Time budget per problem**: _fill in from results_

### Phase 1 preview: Prompt Engineering

Phase 0 showed us the raw baseline. Phase 1 will push further:
- **Chain-of-Thought prompting**: Structured step-by-step templates
- **Tool-Integrated Reasoning (TIR)**: Let the model write and execute Python code mid-reasoning
- **Few-shot examples**: Show 1–3 solved problems before the target problem
- **Early stopping**: Stop generating when 4+ samples converge → faster maj@64

All of this without touching the model weights. Phase 1 is purely prompting.